In [34]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [35]:
load_dotenv()

True

In [36]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [37]:
class Blogstate(TypedDict):
    title:str
    outline:str
    content:str
    score:float

In [38]:
def create_outline(state : Blogstate)->Blogstate:
    title = state['title']

    prompt = f'generate a detailed outline for a blog on the topic of {title}'

    outline = model.invoke(prompt).content
    state['outline'] = outline

    return state



In [39]:
def create_blog(state : Blogstate)->Blogstate:
    title = state['title']
    outline=state['outline']

    prompt = f'Generate a detailed prompt on the title -{title} using the following outline \n {outline}'
    content = model.invoke(prompt).content
    state['content'] = content

    return state

In [40]:
def eval_Score(state:Blogstate)->Blogstate:
    title =state['title']
    outline=state['outline']
    content = state['content']

    prompt = f"""generate a score out of 10 based on this blog's outline  - {outline}  and content -{content} on the topic of {title} """

    score = model.invoke(prompt)

    state['score'] = score

    return state

In [41]:
graph = StateGraph(Blogstate)

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('eval_Score',eval_Score)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','eval_Score')
graph.add_edge('eval_Score',END)

workflow=graph.compile()

In [42]:
initial_state = {'title':'AI'}

finalstate=workflow.invoke(initial_state)

print(finalstate)

{'title': 'AI', 'outline': 'Here\'s a detailed outline for a blog post on the topic of AI, designed to be comprehensive yet accessible for a general audience.\n\n---\n\n## Blog Title Options:\n\n*   **AI Unpacked: Understanding the Revolution That\'s Changing Everything** (Primary Choice)\n*   Demystifying AI: Your Essential Guide to the Future\n*   The AI Era: Exploring Intelligence, Impact, and Innovation\n*   Beyond the Hype: A Clear Look at Artificial Intelligence\n\n---\n\n## I. Introduction (Approx. 150-200 words)\n\n*   **A. Hook:** Start with a relatable scenario demonstrating AI\'s presence in daily life (e.g., Siri/Alexa, Netflix recommendations, self-driving car prototypes, spam filters).\n    *   *Example:* "From the moment your alarm clock rings, to the personalized recommendations on your streaming service, Artificial Intelligence is quietly, yet powerfully, woven into the fabric of our daily lives."\n*   **B. Brief Definition & Significance:** Introduce AI as the simulat

In [43]:
print(finalstate['outline'])

Here's a detailed outline for a blog post on the topic of AI, designed to be comprehensive yet accessible for a general audience.

---

## Blog Title Options:

*   **AI Unpacked: Understanding the Revolution That's Changing Everything** (Primary Choice)
*   Demystifying AI: Your Essential Guide to the Future
*   The AI Era: Exploring Intelligence, Impact, and Innovation
*   Beyond the Hype: A Clear Look at Artificial Intelligence

---

## I. Introduction (Approx. 150-200 words)

*   **A. Hook:** Start with a relatable scenario demonstrating AI's presence in daily life (e.g., Siri/Alexa, Netflix recommendations, self-driving car prototypes, spam filters).
    *   *Example:* "From the moment your alarm clock rings, to the personalized recommendations on your streaming service, Artificial Intelligence is quietly, yet powerfully, woven into the fabric of our daily lives."
*   **B. Brief Definition & Significance:** Introduce AI as the simulation of human intelligence by machines, emphasizi

In [44]:
print(finalstate['content'])

Here's a detailed prompt for a blog post based on your comprehensive outline:

---

**Blog Post Prompt: AI Unpacked: Understanding the Revolution That's Changing Everything**

**Target Audience:** General audience, curious about technology but not necessarily experts.
**Tone:** Informative, accessible, engaging, balanced, and slightly enthusiastic about the future, while also acknowledging challenges.
**Overall Word Count Goal:** Approximately 1600-2000 words.

---

**I. Blog Title (Choose ONE and use it as the H1):**

*   **AI Unpacked: Understanding the Revolution That's Changing Everything** (Primary Choice)
*   Demystifying AI: Your Essential Guide to the Future
*   The AI Era: Exploring Intelligence, Impact, and Innovation
*   Beyond the Hype: A Clear Look at Artificial Intelligence

---

**II. Introduction (Approx. 150-200 words)**

*   **A. Hook:** Start with a relatable, everyday scenario that vividly demonstrates AI's subtle but powerful presence in our daily lives. Think abou

In [45]:
print(finalstate['score'])

content='This blog\'s outline and content prompt is **10/10**.\n\nHere\'s why:\n\n1.  **Exceptional Comprehensiveness:** The outline covers every essential aspect of AI for a general audience, from basic definitions and history to how it works, its vast applications, the critical opportunities and challenges, and thoughtful considerations for the future. Nothing feels overlooked.\n\n2.  **Unwavering Accessibility Focus:**\n    *   **Relatable Hooks:** The introduction immediately grounds AI in daily life, making it relevant from the first sentence.\n    *   **Simplified Explanations:** Complex terms like Machine Learning, Deep Learning, NLP, and Computer Vision are broken down into understandable, non-technical language with clear analogies ("teaching a child by showing examples").\n    *   **Clear Differentiation:** The "Types of AI" section is crucial for a general audience to understand the difference between current "Narrow AI" and hypothetical "General/Superintelligence."\n    *  